# Fase 1: Validación y Consistencia de Datos
Este notebook realiza la validación preliminar de consistencia entre el modelo principal ($K=6$) y el análisis de sensibilidad A.

In [1]:
import pandas as pd
import numpy as np

# 1. Carga de los archivos
path_k6 = '/content/clientes_segmentados_k6.csv'
path_sens = '/content/clientes_cluster_sensibilidad_A.csv'

try:
    df_k6 = pd.read_csv(path_k6)
    df_sens = pd.read_csv(path_sens)
    print("✅ Archivos cargados correctamente.\n")
except Exception as e:
    print(f"❌ Error al cargar los archivos: {e}")

# 2. Comprobación de dimensiones, tipos, nulos y duplicados
print("=== ANÁLISIS DE DATOS: MODELO PRINCIPAL (K6) ===")
print(f"Dimensiones: {df_k6.shape[0]} filas, {df_k6.shape[1]} columnas")
print(f"Columnas: {list(df_k6.columns)}")
print(f"Valores nulos por columna:\n{df_k6.isnull().sum()}\n")
num_duplicados_k6 = df_k6['Num Cliente'].duplicated().sum() if 'Num Cliente' in df_k6.columns else 'Columna no encontrada'
print(f"Duplicados en 'Num Cliente': {num_duplicados_k6}\n")

print("=== ANÁLISIS DE DATOS: ANÁLISIS DE SENSIBILIDAD A ===")
print(f"Dimensiones: {df_sens.shape[0]} filas, {df_sens.shape[1]} columnas")
print(f"Columnas: {list(df_sens.columns)}")
print(f"Valores nulos por columna:\n{df_sens.isnull().sum()}\n")
num_duplicados_sens = df_sens['Num Cliente'].duplicated().sum() if 'Num Cliente' in df_sens.columns else 'Columna no encontrada'
print(f"Duplicados en 'Num Cliente': {num_duplicados_sens}\n")

# 3. Validación de consistencia de clientes
if 'Num Cliente' in df_k6.columns and 'Num Cliente' in df_sens.columns:
    clientes_k6 = set(df_k6['Num Cliente'])
    clientes_sens = set(df_sens['Num Cliente'])

    solo_en_k6 = clientes_k6 - clientes_sens
    solo_en_sens = clientes_sens - clientes_k6
    coincidentes = clientes_k6.intersection(clientes_sens)

    print("=== VALIDACIÓN DE INTEGRIDAD REFERENCIAL ===")
    print(f"Clientes únicos en K6: {len(clientes_k6)}")
    print(f"Clientes únicos en Sensibilidad A: {len(clientes_sens)}")
    print(f"Clientes coincidentes en ambos archivos: {len(coincidentes)}")
    print(f"Clientes presentes solo en K6: {len(solo_en_k6)}")
    print(f"Clientes presentes solo en Sensibilidad: {len(solo_en_sens)}")

    if len(solo_en_k6) > 0 or len(solo_en_sens) > 0:
        print("⚠️ ADVERTENCIA: Existen discrepancias entre los identificadores de cliente de ambos archivos.")
    else:
        print("✅ Excelente: Ambos archivos contienen exactamente el mismo conjunto de clientes.")
else:
    print("❌ Error: La columna 'Num Cliente' debe estar presente en ambos archivos.")

# 4. Comprobación de la estructura de clústeres
print("\n=== DISTRIBUCIÓN DE CLÚSTERES ===")
if 'Cluster_K6' in df_k6.columns:
    clusters_k6_unique = sorted(df_k6['Cluster_K6'].dropna().unique())
    print(f"Clústeres en K6 (Original): {clusters_k6_unique} (Total: {len(clusters_k6_unique)} clústeres)")
    print(df_k6['Cluster_K6'].value_counts(dropna=False).sort_index().to_frame())
else:
    print("❌ 'Cluster_K6' no encontrado en el archivo original.")

print("")
if 'Cluster_Sensibilidad_A' in df_sens.columns:
    clusters_sens_unique = sorted(df_sens['Cluster_Sensibilidad_A'].dropna().unique())
    print(f"Clústeres en Sensibilidad A (Alternativo): {clusters_sens_unique} (Total: {len(clusters_sens_unique)} clústeres)")
    print(df_sens['Cluster_Sensibilidad_A'].value_counts(dropna=False).sort_index().to_frame())
else:
    print("❌ 'Cluster_Sensibilidad_A' no encontrado en el archivo de sensibilidad.")

✅ Archivos cargados correctamente.

=== ANÁLISIS DE DATOS: MODELO PRINCIPAL (K6) ===
Dimensiones: 498863 filas, 12 columnas
Columnas: ['Num Cliente', 'Num_Productos', 'Num_Contratos', 'Fecha_Primera_Creacion', 'Tiene_Luz', 'Tiene_Gas', 'Clasificacion_Suministro', 'Conflicto_Tipo_Cliente', 'Antiguedad_Meses', 'Num_Potencias_Distintas', 'Num_Escalones_Gas_Distintos', 'Cluster_K6']
Valores nulos por columna:
Num Cliente                    0
Num_Productos                  0
Num_Contratos                  0
Fecha_Primera_Creacion         0
Tiene_Luz                      0
Tiene_Gas                      0
Clasificacion_Suministro       0
Conflicto_Tipo_Cliente         0
Antiguedad_Meses               0
Num_Potencias_Distintas        0
Num_Escalones_Gas_Distintos    0
Cluster_K6                     0
dtype: int64

Duplicados en 'Num Cliente': 0

=== ANÁLISIS DE DATOS: ANÁLISIS DE SENSIBILIDAD A ===
Dimensiones: 498863 filas, 2 columnas
Columnas: ['Num Cliente', 'Cluster_Sensibilidad_A']
Valor

## Fase 2: Caracterización de los Clústeres del Modelo Principal (K6)
En esta sección se calculan las métricas estadísticas y distribuciones para los 6 clústeres originales sin añadir asunciones subjetivas sobre satisfacción o rentabilidad.

In [4]:
import pandas as pd
import numpy as np

# --- A. TAMAÑO DEL SEGMENTO ---
total_clientes = len(df_k6)
segmentos = df_k6.groupby('Cluster_K6').size().reset_index(name='Num_Clientes')
segmentos['Porcentaje_Total'] = (segmentos['Num_Clientes'] / total_clientes) * 100
segmentos['Orden_Tamano'] = segmentos['Num_Clientes'].rank(ascending=False, method='min').astype(int)

# --- B. ANTIGÜEDAD ---
antiguedad_stats = df_k6.groupby('Cluster_K6')['Antiguedad_Meses'].agg(
    Mean_Antiguedad='mean',
    Median_Antiguedad='median',
    Std_Antiguedad='std',
    Q1_Antiguedad=lambda x: x.quantile(0.25),
    Q3_Antiguedad=lambda x: x.quantile(0.75),
    Min_Antiguedad='min',
    Max_Antiguedad='max'
).reset_index()

# --- C. CONTRATACIÓN (Estadísticos) ---
contratos_stats = df_k6.groupby('Cluster_K6')['Num_Contratos'].agg(
    Mean_Contratos='mean',
    Median_Contratos='median',
    Std_Contratos='std',
    Min_Contratos='min',
    Max_Contratos='max'
).reset_index()

# --- D. TIPO DE SUMINISTRO (Distribución por Clúster) ---
suministro_dist = pd.crosstab(
    df_k6['Cluster_K6'],
    df_k6['Clasificacion_Suministro'],
    normalize='index'
) * 100
suministro_dist.columns = [f'Suministro_{col}_%' for col in suministro_dist.columns]
suministro_dist = suministro_dist.reset_index()

suministro_counts = pd.crosstab(
    df_k6['Cluster_K6'],
    df_k6['Clasificacion_Suministro']
).reset_index()
suministro_counts.columns = ['Cluster_K6'] + [f'Suministro_{col}_N' for col in suministro_counts.columns[1:]]

# --- E. INDICADORES COMPLEMENTARIOS (Corrección de sintaxis de agregación) ---
complementarios = df_k6.groupby('Cluster_K6').agg({
    'Num_Potencias_Distintas': 'mean',
    'Num_Escalones_Gas_Distintos': 'mean',
    'Conflicto_Tipo_Cliente': lambda x: (x == 1).sum()
}).rename(columns={
    'Num_Potencias_Distintas': 'Mean_Potencias',
    'Num_Escalones_Gas_Distintos': 'Mean_Escalones_Gas',
    'Conflicto_Tipo_Cliente': 'Conflictos_N'
}).reset_index()

complementarios['Conflictos_%'] = (complementarios['Conflictos_N'] / segmentos['Num_Clientes']) * 100

# --- CONSOLIDACIÓN EN LA TABLA MAESTRA ---
tabla_maestra = segmentos.merge(antiguedad_stats, on='Cluster_K6')
tabla_maestra = tabla_maestra.merge(contratos_stats, on='Cluster_K6')
tabla_maestra = tabla_maestra.merge(suministro_dist, on='Cluster_K6')
tabla_maestra = tabla_maestra.merge(complementarios, on='Cluster_K6')

print("=== TABLA MAESTRA DE CLÚSTERES K6 ===")
display(tabla_maestra.round(2))

# Validación de consistencia de totales
print(f"\nSuma total de clientes en clústeres: {tabla_maestra['Num_Clientes'].sum()} (Esperado: {total_clientes})")
print(f"Suma de porcentajes: {tabla_maestra['Porcentaje_Total'].sum():.2f}%")

=== TABLA MAESTRA DE CLÚSTERES K6 ===


,Cluster_K6,Num_Clientes,Porcentaje_Total,Orden_Tamano,Mean_Antiguedad,Median_Antiguedad,Std_Antiguedad,Q1_Antiguedad,Q3_Antiguedad,Min_Antiguedad,...,Std_Contratos,Min_Contratos,Max_Contratos,Suministro_Dual_%,Suministro_Solo Gas_%,Suministro_Solo Luz_%,Mean_Potencias,Mean_Escalones_Gas,Conflictos_N,Conflictos_%
0,0,84100,16.86,2,92.68,92.98,23.30,70.31,121.26,56.74,...,0.00,1,1,0.00,0.00,100.00,1.00,0.00,0,0.00
1,1,77927,15.62,3,32.85,20.01,32.56,6.05,51.58,0.00,...,0.00,1,1,100.00,0.00,0.00,1.00,1.00,175,0.22
2,2,20047,4.02,6,38.69,26.68,35.99,7.33,62.78,0.00,...,0.12,1,5,0.00,100.00,0.00,0.00,1.00,6,0.03
3,3,255561,51.23,1,20.77,17.08,16.14,6.18,32.46,0.00,...,0.00,1,1,0.00,0.00,100.00,1.00,0.00,0,0.00
4,4,33932,6.80,4,42.23,33.54,35.87,10.87,64.57,0.00,...,0.82,2,46,100.00,0.00,0.00,1.20,1.02,321,0.95
5,5,27296,5.47,5,41.12,30.55,35.99,11.27,59.56,0.00,...,1.60,2,121,0.01,0.01,99.99,1.73,0.00,461,1.69



Suma total de clientes en clústeres: 498863 (Esperado: 498863)
Suma de porcentajes: 100.00%


In [3]:
# --- F. DISTRIBUCIONES DETALLADAS ADICIONALES ---
print("=== DISTRIBUCIÓN DETALLADA DE NÚMERO DE CONTRATOS POR CLÚSTER ===")
contratos_detallada = pd.crosstab(df_k6['Cluster_K6'], df_k6['Num_Contratos'])
contratos_detallada_pct = pd.crosstab(df_k6['Cluster_K6'], df_k6['Num_Contratos'], normalize='index') * 100

print("\n--- Cantidades Absolutas (N) ---")
display(contratos_detallada)
print("\n--- Porcentajes por Fila (%) ---")
display(contratos_detallada_pct.round(2))

=== DISTRIBUCIÓN DETALLADA DE NÚMERO DE CONTRATOS POR CLÚSTER ===

--- Cantidades Absolutas (N) ---


Num_Contratos,1,2,3,4,5,6,7,8,9,10,...,35,42,45,46,50,52,54,59,92,121
Cluster_K6,,,,,,,,,,,,,,,,,,,,,
0,84100,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,77927,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,19867,158,16,5,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,255561,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,29993,2947,619,180,74,38,22,10,11,...,0,0,1,1,0,0,0,0,0,0
5,0,21542,3651,1096,423,211,128,74,62,28,...,1,1,0,0,1,1,1,1,1,1



--- Porcentajes por Fila (%) ---


Num_Contratos,1,2,3,4,5,6,7,8,9,10,...,35,42,45,46,50,52,54,59,92,121
Cluster_K6,,,,,,,,,,,,,,,,,,,,,
0,100.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,99.1,0.79,0.08,0.02,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,88.39,8.69,1.82,0.53,0.22,0.11,0.06,0.03,0.03,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,78.92,13.38,4.02,1.55,0.77,0.47,0.27,0.23,0.10,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Fase 3: Auditoría Numérica y Validación de Métricas (K6)
Este script valida de forma independiente y exacta cada métrica obtenida en la fase de caracterización frente a los datos originales para certificar la ausencia de errores o sesgos en el informe final del TFM.

In [7]:
import pandas as pd
import numpy as np

# --- 1. RECALCULO DE ANTIGÜEDAD ---
recalc_antiguedad = df_k6.groupby('Cluster_K6')['Antiguedad_Meses'].agg(
    Recalc_Min='min',
    Recalc_Max='max',
    Recalc_Mean='mean',
    Recalc_Median='median'
).reset_index()

# Comparación con tabla_maestra para Antigüedad
audit_antiguedad = pd.DataFrame({
    'Cluster_K6': recalc_antiguedad['Cluster_K6'],
    'Prev_Min': tabla_maestra['Min_Antiguedad'],
    'Recalc_Min': recalc_antiguedad['Recalc_Min'],
    'Min_Match': np.isclose(tabla_maestra['Min_Antiguedad'], recalc_antiguedad['Recalc_Min']),
    'Prev_Max': tabla_maestra['Max_Antiguedad'],
    'Recalc_Max': recalc_antiguedad['Recalc_Max'],
    'Max_Match': np.isclose(tabla_maestra['Max_Antiguedad'], recalc_antiguedad['Recalc_Max']),
    'Prev_Mean': tabla_maestra['Mean_Antiguedad'],
    'Recalc_Mean': recalc_antiguedad['Recalc_Mean'],
    'Mean_Match': np.isclose(tabla_maestra['Mean_Antiguedad'], recalc_antiguedad['Recalc_Mean']),
    'Prev_Median': tabla_maestra['Median_Antiguedad'],
    'Recalc_Median': recalc_antiguedad['Recalc_Median'],
    'Median_Match': np.isclose(tabla_maestra['Median_Antiguedad'], recalc_antiguedad['Recalc_Median'])
})

print("=== 1. AUDITORÍA DE ANTIGÜEDAD POR CLÚSTER ===")
display(audit_antiguedad.round(4))

# --- 2 & 3 & 4. RECALCULO DE DISTRIBUCIÓN DE CONTRATOS ---
recalc_contratos_mean = df_k6.groupby('Cluster_K6')['Num_Contratos'].mean().reset_index(name='Recalc_Mean_Contratos')

audit_contratos_mean = pd.DataFrame({
    'Cluster_K6': recalc_contratos_mean['Cluster_K6'],
    'Prev_Mean_Contratos': tabla_maestra['Mean_Contratos'],
    'Recalc_Mean_Contratos': recalc_contratos_mean['Recalc_Mean_Contratos'],
    'Mean_Contratos_Match': np.isclose(tabla_maestra['Mean_Contratos'], recalc_contratos_mean['Recalc_Mean_Contratos'])
})

print("\n=== 2. AUDITORÍA DE MEDIAS DE CONTRATOS ===")
display(audit_contratos_mean.round(4))

# Validación detallada de Distribuciones Internas de Contratos para Clústeres 4 y 5
pct_contratos = pd.crosstab(df_k6['Cluster_K6'], df_k6['Num_Contratos'], normalize='index') * 100
count_contratos = pd.crosstab(df_k6['Cluster_K6'], df_k6['Num_Contratos'])

print("\n=== 3. AUDITORÍA DETALLADA DE DISTRIBUCIÓN DE CONTRATOS (CLÚSTERES 4 Y 5) ===")
for c in [4, 5]:
    print(f"\n--- Clúster {c} ---")
    dist_df = pd.DataFrame({
        'Num_Contratos': pct_contratos.columns,
        'Clientes_Count_N': count_contratos.loc[c].values,
        'Porcentaje_Interno_%': pct_contratos.loc[c].values
    })
    # Filtrar solo donde hay clientes contratados para simplificar lectura
    display(dist_df[dist_df['Clientes_Count_N'] > 0].round(4))

# --- 5. RECALCULO DE CONFLICTOS ---
# Agrupación y suma directa de booleanos para evitar conflictos con DataFrames de paso en lambdas
recalc_conflictos = df_k6.groupby('Cluster_K6')['Conflicto_Tipo_Cliente'].apply(lambda x: (x == 1).sum()).reset_index(name='Recalc_Conflictos_N')
recalc_conflictos['Recalc_Conflictos_%'] = (recalc_conflictos['Recalc_Conflictos_N'] / tabla_maestra['Num_Clientes']) * 100

audit_conflictos = pd.DataFrame({
    'Cluster_K6': recalc_conflictos['Cluster_K6'],
    'Prev_Conflictos_N': tabla_maestra['Conflictos_N'],
    'Recalc_Conflictos_N': recalc_conflictos['Recalc_Conflictos_N'],
    'N_Match': tabla_maestra['Conflictos_N'] == recalc_conflictos['Recalc_Conflictos_N'],
    'Prev_Conflictos_%': tabla_maestra['Conflictos_%'],
    'Recalc_Conflictos_%': recalc_conflictos['Recalc_Conflictos_%'],
    'Pct_Match': np.isclose(tabla_maestra['Conflictos_%'], recalc_conflictos['Recalc_Conflictos_%'])
})

print("\n=== 4. AUDITORÍA DE CONFLICTO_TIPO_CLIENTE = 1 ===")
display(audit_conflictos.round(4))

# --- 6. VALIDACIÓN DE TOTALES ---
total_clientes_recalc = tabla_maestra['Num_Clientes'].sum()
print(f"\n=== 5. COMPROBACIÓN DEL VOLUMEN TOTAL ===")
print(f"Suma de clientes de todos los clústeres: {total_clientes_recalc} (Coincidencia exacta con 498.863: {total_clientes_recalc == 498863})")

=== 1. AUDITORÍA DE ANTIGÜEDAD POR CLÚSTER ===


,Cluster_K6,Prev_Min,Recalc_Min,Min_Match,Prev_Max,Recalc_Max,Max_Match,Prev_Mean,Recalc_Mean,Mean_Match,Prev_Median,Recalc_Median,Median_Match
0,0,56.7392,56.7392,True,125.6345,125.6345,True,92.6791,92.6791,True,92.9774,92.9774,True
1,1,0.0000,0.0000,True,131.4825,131.4825,True,32.8452,32.8452,True,20.0082,20.0082,True
2,2,0.0000,0.0000,True,131.4825,131.4825,True,38.6884,38.6884,True,26.6776,26.6776,True
3,3,0.0000,0.0000,True,56.6407,56.6407,True,20.7685,20.7685,True,17.0842,17.0842,True
4,4,0.0000,0.0000,True,131.4825,131.4825,True,42.2262,42.2262,True,33.5441,33.5441,True
5,5,0.0000,0.0000,True,125.3060,125.3060,True,41.1158,41.1158,True,30.5544,30.5544,True



=== 2. AUDITORÍA DE MEDIAS DE CONTRATOS ===


,Cluster_K6,Prev_Mean_Contratos,Recalc_Mean_Contratos,Mean_Contratos_Match
0,0,1.0000,1.0000,True
1,1,1.0000,1.0000,True
2,2,1.0104,1.0104,True
3,3,1.0000,1.0000,True
4,4,2.1799,2.1799,True
5,5,2.4066,2.4066,True



=== 3. AUDITORÍA DETALLADA DE DISTRIBUCIÓN DE CONTRATOS (CLÚSTERES 4 Y 5) ===

--- Clúster 4 ---


...


--- Clúster 5 ---


,Num_Contratos,Clientes_Count_N,Porcentaje_Interno_%
1,2,21542,78.9200
2,3,3651,13.3756
3,4,1096,4.0152
4,5,423,1.5497
5,6,211,0.7730
6,7,128,0.4689
7,8,74,0.2711
8,9,62,0.2271
9,10,28,0.1026
10,11,16,0.0586



=== 4. AUDITORÍA DE CONFLICTO_TIPO_CLIENTE = 1 ===


,Cluster_K6,Prev_Conflictos_N,Recalc_Conflictos_N,N_Match,Prev_Conflictos_%,Recalc_Conflictos_%,Pct_Match
0,0,0,0,True,0.0000,0.0000,True
1,1,175,175,True,0.2246,0.2246,True
2,2,6,6,True,0.0299,0.0299,True
3,3,0,0,True,0.0000,0.0000,True
4,4,321,321,True,0.9460,0.9460,True
5,5,461,461,True,1.6889,1.6889,True



=== 5. COMPROBACIÓN DEL VOLUMEN TOTAL ===
Suma de clientes de todos los clústeres: 498863 (Coincidencia exacta con 498.863: True)


## Fase 4: Comparación entre el Modelo Principal (K6) y el Análisis de Sensibilidad A
En esta sección se realiza el cruce de las clasificaciones para analizar la estabilidad de las particiones al retirar el tipo de suministro del espacio de variables.

In [8]:
import pandas as pd
import numpy as np

# 1. Alineación de los conjuntos de datos por Num Cliente
df_comparacion = pd.merge(
    df_k6[['Num Cliente', 'Cluster_K6', 'Antiguedad_Meses', 'Num_Contratos', 'Clasificacion_Suministro']],
    df_sens[['Num Cliente', 'Cluster_Sensibilidad_A']],
    on='Num Cliente',
    how='inner'
)

# 2. Construcción de la Tabla Cruzada (Frecuencias Absolutas N)
tabla_cruzada_n = pd.crosstab(
    df_comparacion['Cluster_K6'],
    df_comparacion['Cluster_Sensibilidad_A'],
    margins=True,
    margins_name='Total'
)

# 3. Construcción de la Tabla Cruzada (Porcentajes por fila)
tabla_cruzada_pct = pd.crosstab(
    df_comparacion['Cluster_K6'],
    df_comparacion['Cluster_Sensibilidad_A'],
    normalize='index'
) * 100

print("=== TABLA CRUZADA (CANTIDAD DE CLIENTES N) ===")
display(tabla_cruzada_n)

print("\n=== TABLA CRUZADA (DISTRIBUCIÓN % POR FILA DEL CLÚSTER ORIGINAL) ===")
display(tabla_cruzada_pct.round(2))

# 4. Caracterización Descriptiva de los Clústeres del Modelo de Sensibilidad A
sens_carac = df_comparacion.groupby('Cluster_Sensibilidad_A').agg({
    'Num Cliente': 'count',
    'Antiguedad_Meses': ['mean', 'median', 'min', 'max'],
    'Num_Contratos': ['mean', 'median', 'min', 'max']
})
sens_carac.columns = [
    'Clientes_N',
    'Antiguedad_Media', 'Antiguedad_Mediana', 'Antiguedad_Min', 'Antiguedad_Max',
    'Contratos_Media', 'Contratos_Mediana', 'Contratos_Min', 'Contratos_Max'
]
sens_carac = sens_carac.reset_index()
sens_carac['Porcentaje_Total'] = (sens_carac['Clientes_N'] / len(df_comparacion)) * 100

print("\n=== CARACTERIZACIÓN DESCRIPTIVA DE LOS CLÚSTERES DE SENSIBILIDAD A ===")
display(sens_carac.round(2))

# Validación de suma total
print(f"\nComprobación de volumen total cruzado: {tabla_cruzada_n.loc['Total', 'Total']} (Esperado: 498863)")

=== TABLA CRUZADA (CANTIDAD DE CLIENTES N) ===


Cluster_Sensibilidad_A,0,1,2,3,4,5,Total
Cluster_K6,,,,,,,
0,0,0,50397,0,0,33703,84100
1,48453,0,8454,0,0,21020,77927
2,10947,116,3081,43,21,5839,20047
3,192399,0,0,0,0,63162,255561
4,0,21212,0,9337,3383,0,33932
5,0,15612,0,6424,5260,0,27296
Total,251799,36940,61932,15804,8664,123724,498863



=== TABLA CRUZADA (DISTRIBUCIÓN % POR FILA DEL CLÚSTER ORIGINAL) ===


Cluster_Sensibilidad_A,0,1,2,3,4,5
Cluster_K6,,,,,,
0,0.00,0.00,59.93,0.00,0.00,40.07
1,62.18,0.00,10.85,0.00,0.00,26.97
2,54.61,0.58,15.37,0.21,0.10,29.13
3,75.28,0.00,0.00,0.00,0.00,24.72
4,0.00,62.51,0.00,27.52,9.97,0.00
5,0.00,57.20,0.00,23.53,19.27,0.00



=== CARACTERIZACIÓN DESCRIPTIVA DE LOS CLÚSTERES DE SENSIBILIDAD A ===


,Cluster_Sensibilidad_A,Clientes_N,Antiguedad_Media,Antiguedad_Mediana,Antiguedad_Min,Antiguedad_Max,Contratos_Media,Contratos_Mediana,Contratos_Min,Contratos_Max,Porcentaje_Total
0,0,251799,12.62,10.45,0.00,32.69,1.00,1.0,1,1,50.47
1,1,36940,22.56,18.86,0.00,57.33,2.00,2.0,2,2,7.40
2,2,61932,108.24,112.20,80.53,131.48,1.00,1.0,1,1,12.41
3,3,15804,92.04,92.71,57.43,131.48,2.07,2.0,2,3,3.17
4,4,8664,31.65,25.23,0.00,130.86,3.87,3.0,3,121,1.74
5,5,123724,52.94,50.79,32.72,80.49,1.00,1.0,1,1,24.80



Comprobación de volumen total cruzado: 498863 (Esperado: 498863)


### Auditoría Estadística del Modelo de Sensibilidad A
Este script calcula de forma directa las estadísticas descriptivas de los clústeres de sensibilidad para verificar su consistencia interna y auditar posibles discrepancias numéricas.

In [9]:
import pandas as pd
import numpy as np

# Asegurar alineación limpia de los datos mediante inner join por Num Cliente
df_auditoria_sens = pd.merge(
    df_k6[['Num Cliente', 'Antiguedad_Meses', 'Num_Contratos']],
    df_sens[['Num Cliente', 'Cluster_Sensibilidad_A']],
    on='Num Cliente',
    how='inner'
)

total_clientes_audit = len(df_auditoria_sens)
print(f"Volumen total de registros para auditoría: {total_clientes_audit} (Esperado: 498863)\n")

# 1. ESTADÍSTICOS DE ANTIGÜEDAD POR CLÚSTER (SENSIBILIDAD A)
antiguedad_sens_stats = df_auditoria_sens.groupby('Cluster_Sensibilidad_A')['Antiguedad_Meses'].agg([
    ('Clientes_N', 'count'),
    ('Media_Antiguedad', 'mean'),
    ('Mediana_Antiguedad', 'median'),
    ('Desv_Est_Antiguedad', 'std'),
    ('Min_Antiguedad', 'min'),
    ('Q1_Antiguedad', lambda x: x.quantile(0.25)),
    ('Q3_Antiguedad', lambda x: x.quantile(0.75)),
    ('Max_Antiguedad', 'max')
]).reset_index()

antiguedad_sens_stats['Porcentaje_Total_%'] = (antiguedad_sens_stats['Clientes_N'] / total_clientes_audit) * 100

# Reordenar columnas para visualización óptima
antiguedad_sens_stats = antiguedad_sens_stats[[
    'Cluster_Sensibilidad_A', 'Clientes_N', 'Porcentaje_Total_%',
    'Media_Antiguedad', 'Mediana_Antiguedad', 'Desv_Est_Antiguedad',
    'Min_Antiguedad', 'Q1_Antiguedad', 'Q3_Antiguedad', 'Max_Antiguedad'
]]

# 2. ESTADÍSTICOS DE CONTRATOS POR CLÚSTER (SENSIBILIDAD A)
contratos_sens_stats = df_auditoria_sens.groupby('Cluster_Sensibilidad_A')['Num_Contratos'].agg([
    ('Media_Contratos', 'mean'),
    ('Mediana_Contratos', 'median'),
    ('Desv_Est_Contratos', 'std'),
    ('Min_Contratos', 'min'),
    ('Max_Contratos', 'max')
]).reset_index()

# 3. CONTEOS AGRUPADOS DE CONTRATOS (1, 2, 3, 4+ contratos)
def contar_por_rango(group):
    c1 = (group == 1).sum()
    c2 = (group == 2).sum()
    c3 = (group == 3).sum()
    c4_mas = (group >= 4).sum()
    tot = len(group)
    return pd.Series({
        '1_Contrato_N': c1, '1_Contrato_%': (c1/tot)*100,
        '2_Contratos_N': c2, '2_Contratos_%': (c2/tot)*100,
        '3_Contratos_N': c3, '3_Contratos_%': (c3/tot)*100,
        '4+_Contratos_N': c4_mas, '4+_Contratos_%': (c4_mas/tot)*100
    })

contratos_grupos = df_auditoria_sens.groupby('Cluster_Sensibilidad_A')['Num_Contratos'].apply(contar_por_rango).unstack().reset_index()

# 4. TABLA CONSOLIDADA DE LA AUDITORÍA DE SENSIBILIDAD A
tabla_consolidada_sens = pd.merge(antiguedad_sens_stats, contratos_sens_stats, on='Cluster_Sensibilidad_A')
tabla_consolidada_sens = pd.merge(tabla_consolidada_sens, contratos_grupos, on='Cluster_Sensibilidad_A')

print("=== TABLA MAESTRA DE AUDITORÍA: MODELO SENSIBILIDAD A ===")
display(tabla_consolidada_sens.round(4))

# 5. DETALLE DE LA DISTRIBUCIÓN COMPLETA DE CONTRATOS (Para verificar Clúster 0 y otros)
print("\n=== DISTRIBUCIÓN COMPLETA DE CONTRATOS POR CLÚSTER (SENSIBILIDAD A) ===")
dist_completa_n = pd.crosstab(df_auditoria_sens['Cluster_Sensibilidad_A'], df_auditoria_sens['Num_Contratos'])
dist_completa_pct = pd.crosstab(df_auditoria_sens['Cluster_Sensibilidad_A'], df_auditoria_sens['Num_Contratos'], normalize='index') * 100

print("--- Cantidades Absolutas (N) ---")
display(dist_completa_n)

print("\n--- Porcentajes por Fila (%) ---")
display(dist_completa_pct.round(4))

# Comprobación de integridad del total
suma_total_clientes = tabla_consolidada_sens['Clientes_N'].sum()
print(f"\nVerificación de volumen total: {suma_total_clientes} (¿Coincide exactamente con 498,863?: {suma_total_clientes == 498863})")

Volumen total de registros para auditoría: 498863 (Esperado: 498863)

=== TABLA MAESTRA DE AUDITORÍA: MODELO SENSIBILIDAD A ===


,Cluster_Sensibilidad_A,Clientes_N,Porcentaje_Total_%,Media_Antiguedad,Mediana_Antiguedad,Desv_Est_Antiguedad,Min_Antiguedad,Q1_Antiguedad,Q3_Antiguedad,Max_Antiguedad,...,Min_Contratos,Max_Contratos,1_Contrato_N,1_Contrato_%,2_Contratos_N,2_Contratos_%,3_Contratos_N,3_Contratos_%,4+_Contratos_N,4+_Contratos_%
0,0,251799,50.4746,12.6165,10.4476,9.3254,0.0000,4.3368,19.6140,32.6899,...,1,1,251799.0,100.0,0.0,0.0000,0.0,0.0000,0.0,0.0000
1,1,36940,7.4048,22.5569,18.8583,17.0036,0.0000,6.9322,36.7310,57.3306,...,2,2,0.0,0.0,36940.0,100.0000,0.0,0.0000,0.0,0.0000
2,2,61932,12.4146,108.2391,112.1971,13.4917,80.5257,97.4456,121.2649,131.4825,...,1,1,61932.0,100.0,0.0,0.0000,0.0,0.0000,0.0,0.0000
3,3,15804,3.1680,92.0411,92.7146,22.2861,57.4292,70.8337,114.1027,131.4825,...,2,3,0.0,0.0,14753.0,93.3498,1051.0,6.6502,0.0,0.0000
4,4,8664,1.7367,31.6522,25.2320,26.9073,0.0000,8.5339,49.6099,130.8583,...,3,121,0.0,0.0,0.0,0.0000,5563.0,64.2082,3101.0,35.7918
5,5,123724,24.8012,52.9385,50.7926,13.1575,32.7228,42.2505,63.3101,80.4928,...,1,1,123724.0,100.0,0.0,0.0000,0.0,0.0000,0.0,0.0000



=== DISTRIBUCIÓN COMPLETA DE CONTRATOS POR CLÚSTER (SENSIBILIDAD A) ===
--- Cantidades Absolutas (N) ---


Num_Contratos,1,2,3,4,5,6,7,8,9,10,...,35,42,45,46,50,52,54,59,92,121
Cluster_Sensibilidad_A,,,,,,,,,,,,,,,,,,,,,
0,251799,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,36940,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,61932,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,14753,1051,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,5563,1720,604,285,166,96,72,39,...,1,1,1,1,1,1,1,1,1,1
5,123724,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



--- Porcentajes por Fila (%) ---


Num_Contratos,1,2,3,4,5,6,7,8,9,10,...,35,42,45,46,50,52,54,59,92,121
Cluster_Sensibilidad_A,,,,,,,,,,,,,,,,,,,,,
0,100.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.0,100.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,0.0,93.3498,6.6502,0.0000,0.0000,0.0000,0.000,0.000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.0,0.0000,64.2082,19.8523,6.9714,3.2895,1.916,1.108,0.831,0.4501,...,0.0115,0.0115,0.0115,0.0115,0.0115,0.0115,0.0115,0.0115,0.0115,0.0115
5,100.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.000,0.000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000



Verificación de volumen total: 498863 (¿Coincide exactamente con 498,863?: True)


## 4. Interpretación Definitiva de los Clústeres Originales (K6)

En esta sección se presenta la caracterización descriptiva final de la segmentación obtenida con el modelo $K$-Means original ($K=6$). Este análisis constituye la base empírica para el Capítulo 7 (Resultados) y el punto de partida teórico para formular las hipótesis de negocio que estructurarán el Capítulo 8 (Propuestas de validación comercial) del TFM.

### 4.1. Tabla Maestra Comparativa de los Clústeres Originales ($K6$)

La siguiente tabla contiene las métricas estadísticas calculadas y auditadas directamente sobre el conjunto de datos integrado por los $498,863$ registros:

| Clúster | Denominación Descriptiva Neutral | Clientes ($N$) | % Total | Suministro Principal | Antigüedad Media (Meses) | Antigüedad Mediana (Meses) | Contratos Medios | Contratos Medianos | Potencias Medias | Escalones Gas Medios | Conflictos ($N$) | Conflictos (% Interno) |
| :---: | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **C0** | *Suministro Eléctrico Exclusivo con Registro Temporal Alto* | 84,100 | 16.86% | Solo Luz ($100.00\%$) | 92.68 | 92.98 | 1.00 | 1.0 | 1.00 | 0.00 | 0 | 0.00% |
| **C1** | *Suministro Dual Exclusivo con Registro Temporal Corto-Medio y Contrato Único* | 77,927 | 15.62% | Dual ($100.00\%$) | 32.85 | 20.01 | 1.00 | 1.0 | 1.00 | 1.00 | 175 | 0.22% |
| **C2** | *Clientes clasificados como Suministro de Gas Exclusivo con Contrato Único* | 20,047 | 4.02% | Solo Gas ($100.00\%$) | 38.69 | 26.68 | 1.01 | 1.0 | 0.00 | 1.00 | 6 | 0.03% |
| **C3** | *Suministro Eléctrico Exclusivo con Registro Temporal Corto* | 255,561 | 51.23% | Solo Luz ($100.00\%$) | 20.77 | 17.08 | 1.00 | 1.0 | 1.00 | 0.00 | 0 | 0.00% |
| **C4** | *Suministro Dual con Registro Temporal Medio y Contratos Múltiples* | 33,932 | 6.80% | Dual ($100.00\%$) | 42.23 | 33.54 | 2.18 | 2.0 | 1.20 | 1.02 | 321 | 0.95% |
| **C5** | *Suministro Eléctrico con Registro Temporal Medio y Alta Densidad de Contratos* | 27,296 | 5.47% | Solo Luz ($99.99\%$) | 41.12 | 30.55 | 2.41 | 2.0 | 1.73 | 0.00 | 461 | 1.69% |

### 4.2. Fichas Descriptivas por Clúster

#### Clúster 0 (C0): Suministro Eléctrico Exclusivo con Registro Temporal Alto
* **Volumen:** 84,100 clientes ($16.86\%$ del total).
* **Suministro:** Monoproducto eléctrico puro ($100.00\%$ bajo la categoría `Solo Luz`).
* **Registro Temporal:** Presenta un historial de registro alto, con una media de **92.68 meses** y una mediana de **92.98 meses** (rango de 56.74 a 125.63 meses).
* **Estructura de Contratación:** Estructura de contrato único. Tanto la media como la mediana se sitúan en **1.00 contrato**.
* **Parámetros Técnicos:** Media de **1.00 potencia eléctrica contratada** y ningún escalón de gas asignado.
* **Incidencias:** **0 conflictos** comerciales registrados.

#### Clúster 1 (C1): Suministro Dual Exclusivo con Registro Temporal Corto-Medio y Contrato Único
* **Volumen:** 77,927 clientes ($15.62\%$ del total).
* **Suministro:** Dualidad de servicios pura ($100.00\%$ bajo la categoría `Dual`).
* **Registro Temporal:** Historial corto-medio, con una media de **32.85 meses** y una mediana de **20.01 meses** (rango de 0.0 a 131.48 meses).
* **Estructura de Contratación:** Estructura de contrato único. Tanto la media como la mediana se sitúan en **1.00 contrato**.
* **Parámetros Técnicos:** Media de **1.00 potencia eléctrica contratada** y **1.00 escalón de gas** registrado.
* **Incidencias:** Registra **175 conflictos** comerciales ($0.22\%$ del segmento).

#### Clúster 2 (C2): Clientes clasificados como Suministro de Gas Exclusivo con Contrato Único
* **Volumen:** 20,047 clientes ($4.02\%$ del total). Representa el menor grupo volumétrico del modelo.
* **Suministro:** Monoproducto de gas puro ($100.00\%$ bajo la categoría `Solo Gas`).
* **Registro Temporal:** Historial intermedio, con una media de **38.69 meses** y una mediana de **26.68 meses** (rango de 0.0 a 131.48 meses).
* **Estructura de Contratación:** Patrón predominantemente de contrato único, con una media de **1.01 contratos** y mediana de **1.0**. El $99.10\%$ de los clientes de este grupo posee exactamente 1 contrato.
* **Parámetros Técnicos:** Media de **1.00 escalón de gas** contratado y ninguna potencia eléctrica.
* **Incidencias:** Registra **6 conflictos** comerciales ($0.03\%$ del segmento).

#### Clúster 3 (C3): Suministro Eléctrico Exclusivo con Registro Temporal Corto
* **Volumen:** 255,561 clientes ($51.23\%$ del total). Es la mayoría absoluta de la cartera de clientes.
* **Suministro:** Monoproducto eléctrico puro ($100.00\%$ bajo la categoría `Solo Luz`).
* **Registro Temporal:** Representa la distribución de antigüedad más baja del portafolio, con una media de **20.77 meses** y una mediana de **17.08 meses** (con un límite superior estricto de 56.64 meses).
* **Estructura de Contratación:** Estructura de contrato único. Tanto la media como la mediana se sitúan en **1.00 contrato**.
* **Parámetros Técnicos:** Media de **1.00 potencia eléctrica contratada** y ningún escalón de gas registrado.
* **Incidencias:** **0 conflictos** comerciales registrados.

#### Clúster 4 (C4): Suministro Dual con Registro Temporal Medio y Contratos Múltiples
* **Volumen:** 33,932 clientes ($6.80\%$ del total).
* **Suministro:** Dualidad de servicios pura ($100.00\%$ bajo la categoría `Dual`).
* **Registro Temporal:** Historial intermedio, con una media de **42.23 meses** y una mediana de **33.54 meses** (rango de 0.0 a 131.48 meses).
* **Estructura de Contratación:** Estructura de contratos múltiples. La media de contratos se sitúa en **2.18** y la mediana en **2.0**. El $100.00\%$ de los registros cuenta con 2 o más contratos (el $88.39\%$ posee exactamente 2 contratos).
* **Parámetros Técnicos:** Promedio de **1.20 potencias contratadas** y **1.02 escalones de gas** registrados por cliente.
* **Incidencias:** Registra **321 conflictos** comerciales ($0.95\%$ del segmento).

#### Clúster 5 (C5): Suministro Eléctrico con Registro Temporal Medio y Alta Densidad de Contratos
* **Volumen:** 27,296 clientes ($5.47\%$ del total).
* **Suministro:** Monoproducto eléctrico casi total ($99.99\%$ de registros bajo la categoría `Solo Luz`).
* **Registro Temporal:** Historial intermedio, con una media de **41.12 meses** y una mediana de **30.55 meses** (rango de 0.0 a 125.31 meses).
* **Estructura de Contratación:** Mayor densidad contractual registrada en el modelo original, con una media de **2.41 contratos** y mediana de **2.0** (rango de 2 a 121 contratos). El $100.00\%$ cuenta con 2 o más contratos.
* **Parámetros Técnicos:** Muestra la mayor variabilidad eléctrica, con una media de **1.73 potencias contratadas** por cliente.
* **Incidencias:** Presenta el volumen absoluto y relativo más alto de conflictos, sumando **461 registros** ($1.69\%$ del segmento).

### 4.3. Síntesis Comparativa de los Segmentos Clave

* **Diferenciación en Suministros Monoproducto de Luz (C0 frente a C3):**
  Ambos clústeres se componen exclusivamente de clientes con suministro de electricidad y un único contrato. Sin embargo, se diferencian claramente por sus curvas de antigüedad: **C0 concentra los registros más antiguos** de la base (con una mediana de 92.98 meses y un rango mínimo que inicia en 56.74 meses), mientras que **C3 concentra los registros de menor duración** (con mediana de 17.08 meses y un rango máximo topado en 56.64 meses).

* **Diferenciación en Suministros Duales (C1 frente a C4):**
  Aunque ambos grupos abarcan únicamente clientes clasificados como Duales y poseen perfiles temporales de antigüedad mediana cercanos (20.01 meses en C1 frente a 33.54 meses en C4), sus estructuras contractuales son totalmente diferentes: **C1 es un segmento de contrato único** (todos sus clientes tienen un solo contrato consolidado), mientras que **C4 presenta un patrón multicontrato** (media de 2.18 contratos, donde el $100\%$ de sus integrantes registra un mínimo de 2 y hasta un máximo de 46 contratos).

* **Caracterización de C5 (Multicontrato Eléctrico):**
  Este segmento destaca por registrar el mayor volumen medio de contratos por cliente (2.41) y la mayor diversidad en potencia contratada (media de 1.73 potencias distintas). Desde el punto de vista operativo, este grupo experimenta la mayor incidencia interna de conflictos registrados con un $1.69\%$. No disponemos de datos sobre el tamaño de la empresa o consumo eléctrico para determinar si pertenecen a un segmento institucional o corporativo, por lo que su descripción queda acotada estrictamente a su alta densidad contractual.

* **Caracterización de C2 (Suministro de Gas):**
  Este clúster está compuesto de forma exclusiva por clientes clasificados como 'Solo Gas' y con un contrato único. Es importante destacar que esta separación responde al diseño del modelo, el cual incluyó la variable categórica de tipo de suministro. No se asume que este grupo contenga al total absoluto de clientes que consumen gas de la organización fuera del marco de este modelo de segmentación.

### 4.4. Contraste Metodológico con el Análisis de Sensibilidad A

El análisis de sensibilidad A constituye un ejercicio metodológico de validación para entender el comportamiento de la frontera métrica de los clústeres. Al excluir la variable de entrada `Clasificacion_Suministro` (que había sido procesada mediante One-Hot Encoding), la estructura de agrupamiento experimenta una reconfiguración notable:

1. **Condicionamiento por Diseño:** En el modelo original ($K6$), la variable de suministro actuó como un vector de atracción categórica debido a la codificación binaria de sus columnas, lo que determinó la conformación de grupos puros de 'Solo Luz', 'Solo Gas' y 'Dual'. El análisis de sensibilidad demuestra que este comportamiento estuvo condicionado por la selección y codificación de las variables de entrada del modelo original, pero no determina por sí solo que uno de los modelos sea superior al otro, sino que responden a espacios métricos de distancias distintos (un espacio mixto categórico-numérico frente a un espacio puramente numérico).
2. **Dinámica de Redistribución según la Tabla Cruzada:**
   * Las cuentas eléctricas de baja antigüedad de **C3 ($75.28\%$)** y las cuentas duales de baja antigüedad de **C1 ($62.18\%$)** se unifican bajo el **Clúster 0 de Sensibilidad A** ($N=251,799$), respondiendo a su similitud en el eje temporal de baja antigüedad y patrón monocontrato.
   * Los registros de alta antigüedad de **C0 ($59.93\%$)** migran de manera conjunta al **Clúster 2 de Sensibilidad A** ($N=61,932$), priorizando la variable temporal de registro sobre el tipo de suministro.
   * Los clústeres multicontrato **C4 y C5** se reagrupan en los **Clústeres 1, 3 y 4 de Sensibilidad A**, los cuales pasan a segmentar a los clientes basándose estrictamente en intervalos de volumen de contratos (2 contratos, 2-3 contratos, y 4 o más contratos, respectivamente).

*Nota de control:* Se debe evitar tratar las etiquetas del modelo de sensibilidad A como equivalentes directos de los clústeres originales, ya que representan soluciones en espacios de dimensiones distintas.

### 4.5. Limitaciones Metodológicas de la Interpretación

De acuerdo con los estándares de rigor de la investigación académica, se documentan las siguientes limitaciones empíricas asociadas a este análisis:

* **Influencia de la Selección de Variables:** La presencia de clústeres segregados estrictamente por tipo de servicio en el modelo original es consecuencia directa de la inclusión y ponderación de la variable `Clasificacion_Suministro` codificada en OHE.
* **Ausencia de Atributos de Negocio Críticos:** No se dispone en los archivos analizados de variables sobre **facturación acumulada, márgenes financieros, consumo real de energía (kWh), registros de bajas efectivas, ni respuesta histórica a campañas de marketing o satisfacción**.
* **Imposibilidad de Inferir Causalidad:** La presencia de mayor antigüedad en clústeres como C0 o C2 no permite deducir empíricamente la fidelidad, lealtad o satisfacción del cliente, ya que estas distribuciones temporales pueden deberse a factores de inercia o barreras de salida no medidas. Del mismo modo, una mayor densidad de contratos en C5 no es sinónimo automático de rentabilidad o valor comercial.
* **Presencia de Outliers en Contratación:** El valor máximo de **121 contratos** en el Clúster 5 influye de forma directa en las métricas de dispersión de dicho grupo. Este valor debe describirse con prudencia como un caso extremo de contratación múltiple, sin presuponer características de gran escala o naturaleza institucional sin variables auxiliares que lo demuestren.

## 8. Diseño de Estrategias Comerciales Personalizadas por Clúster

### 8.1. Introducción y Objetivo del Capítulo

El objetivo de este capítulo es explorar cómo los resultados taxonómicos de la segmentación realizada con el algoritmo $K$-Means ($K=6$) en el Capítulo 7 pueden servir como marco de referencia para el diseño de **líneas de actuación comerciales personalizadas**. En lugar de aplicar un enfoque comercial homogéneo para toda la cartera, la diferenciación de los clientes sobre la base de sus características de contratación, tipo de suministro y antigüedad de registro permite esbozar propuestas específicas.

Es fundamental destacar que las estrategias presentadas en este capítulo se formulan como **líneas de actuación potenciales y exploratorias**. Su viabilidad técnica, operativa y financiera no está demostrada con los datos disponibles, por lo que su implantación real queda estrictamente condicionada a la validación preliminar de datos comerciales adicionales (tales como consumos, márgenes reales, elegibilidad técnica y coberturas del servicio) que permitan evaluar su rentabilidad real de forma científica.

### 8.2. Estrategias Comerciales Personalizadas para los Seis Clústeres

#### Clúster 0 (C0): Suministro Eléctrico Exclusivo con Registro Temporal Alto
* **Características Observadas en los Datos:** Este segmento abarca a $84,100$ registros ($16.86\%$ del total de la cartera), caracterizados por disponer exclusivamente de suministro de electricidad (`Solo Luz` = $100.00\%$) asociado a un único contrato por cliente, y con la permanencia temporal registrada más elevada del portafolio (mediana de **92.98 meses**; media de **92.68 meses**), con una ausencia total de discrepancias de clasificación de tipo de cliente.
* **Oportunidad Comercial Potencial:** La prolongada permanencia registrada abre la posibilidad teórica de evaluar el interés del cliente por contratar un suministro adicional (gas natural) o servicios complementarios de protección de pagos o asistencia técnica.
* **Líneas de Actuación Propuestas:**
  * *Estudio de Receptividad:* Diseñar sondeos exploratorios de servicios de asistencia o mantenimiento eléctrico asociados al punto de suministro.
  * *Evaluación de Conversión a Dual:* Ofrecer información sobre las condiciones de comercialización de gas natural de la compañía para clientes que ya disponen de electricidad.
* **Información Adicional Necesaria para la Validación:** Consumo real de energía eléctrica (kWh), existencia de acometida y cobertura técnica de la red de gas natural en el punto de suministro, tarifas de la competencia y margen comercial bruto proyectado de la operación.

---

#### Clúster 1 (C1): Suministro Dual con Registro Temporal Corto-Medio y Contrato Único
* **Características Observadas en los Datos:** Compuesto por $77,927$ registros ($15.62\%$ del total), definidos por contar con una clasificación de suministro dual (`Dual` = $100.00\%$), con un único contrato en los registros analizados y una antigüedad intermedia (mediana de **20.01 meses**; media de **32.85 meses**).
* **Oportunidad Comercial Potencial:** Al estar clasificados como clientes duales, la oportunidad reside en afianzar su relación comercial mediante la propuesta de servicios complementarios específicos adecuados a este perfil de doble suministro.
* **Líneas de Actuación Propuestas:**
  * *Sondeo de Servicios Complementarios Duales:* Estudiar la viabilidad de proponer soluciones o informes informativos sobre ahorro energético combinado, eficiencia en el consumo de gas y luz, o servicios de asistencia dual.
* **Información Adicional Necesaria para la Validación:** Curvas de carga eléctrica horaria y consumos estacionales de gas, características físicas de la vivienda o local, márgenes financieros consolidados de la cuenta dual y costes de emisión de campañas.

---

#### Clúster 2 (C2): Suministro de Gas Exclusivo con Contrato Único
* **Características Observadas en los Datos:** Es el grupo más reducido con $20,047$ registros ($4.02\%$ del total), definidos por una clasificación exclusiva de suministro de gas natural (`Solo Gas` = $100.00\%$), una estructura predominantemente de contrato único (media de **1.01 contratos**) y una antigüedad registrada media de **38.69 meses** (mediana de **26.68 meses**).
* **Oportunidad Comercial Potencial:** La limitación del servicio actual al vector de gas abre la oportunidad potencial de explorar la captación del suministro eléctrico asociado a dicha cuenta.
* **Líneas de Actuación Propuestas:**
  * *Propuesta de Captación Eléctrica:* Diseñar ofertas personalizadas para invitar al cliente a dar de alta o trasladar su contrato de electricidad con la compañía.
* **Información Adicional Necesaria para la Validación:** Verificación de la elegibilidad del punto de suministro de electricidad (CUPS activo y compatible), Boletín de Instalación Eléctrica vigente, consumos históricos estimables y costes de adquisición de cliente (CAC) admisibles para este segmento.

---

#### Clúster 3 (C3): Suministro Eléctrico Exclusivo con Registro Temporal Corto
* **Características Observadas en los Datos:** Es la mayoría absoluta de la cartera de clientes, con $255,561$ registros ($51.23\%$ del total), caracterizados por tener suministro eléctrico exclusivo (`Solo Luz` = $100.00\%$), un único contrato asignado y la menor antigüedad media en los registros (mediana de **17.08 meses**; media de **20.77 meses**).
* **Oportunidad Comercial Potencial:** Dada la gran dimensión de este segmento y su corta permanencia en el registro, la oportunidad comercial se enfoca en asegurar una correcta experiencia inicial de contratación y explorar la venta cruzada del suministro de gas natural.
* **Líneas de Actuación Propuestas:**
  * *Optimización de la Comunicación Inicial:* Reforzar los canales de información y bienvenida digital para aclarar dudas sobre los primeros periodos de facturación.
  * *Estudio de Venta Cruzada de Gas:* Analizar la receptividad hacia ofertas de suministro de gas asociadas a la póliza de luz activa.
* **Información Adicional Necesaria para la Validación:** Disponibilidad real de canalización de gas natural en los códigos postales del segmento, canal de captación original, volumen de consumo eléctrico y costes operativos de los canales de comunicación preferidos.

---

#### Clúster 4 (C4): Suministro Dual con Registro Temporal Medio y Contratos Múltiples
* **Características Observadas en los Datos:** Agrupa a $33,932$ registros ($6.80\%$ del total) con clasificación dual (`Dual` = $100.00\%$), una antigüedad registrada media de **42.23 meses** (mediana de **33.54 meses**) y un patrón de contratación múltiple (media de **2.18 contratos** por cliente, donde el $100.00\%$ posee 2 o más contratos).
* **Oportunidad Comercial Potencial:** Al registrar el cliente múltiples contratos con clasificación dual, existe la oportunidad de explorar alternativas de gestión administrativa coordinada que simplifiquen su relación con la compañía.
* **Líneas de Actuación Propuestas:**
  * *Estudio de Gestión Coordinada:* Evaluar la conveniencia de ofrecer un canal de interlocución unificado para la gestión operativa y consultas de sus diferentes pólizas.
  * *Exploración de Simplificación de Facturación:* Estudiar la viabilidad de unificar el envío o el procesamiento administrativo de sus facturas de forma periódica.
* **Información Adicional Necesaria para la Validación:** Viabilidad técnica, contractual y jurídica en los sistemas comerciales para consolidar o coordinar cobros de diferentes puntos de suministro, coincidencia legal de la titularidad de los contratos y costes de adaptación de sistemas frente al margen comercial agregado de la cuenta.

---

#### Clúster 5 (C5): Suministro Eléctrico con Registro Temporal Medio y Alta Densidad de Contratos
* **Características Observadas en los Datos:** Formado por $27,296$ registros ($5.47\%$ del total), mayoritariamente de electricidad (`Solo Luz` = $99.99\%$) con la mayor densidad de contratación media registrada (media de **2.41 contratos** por cliente y un caso extremo de hasta 121 contratos). Registran un promedio de **1.73 potencias contratadas** distintas y la mayor proporción de inconsistencias de clasificación de tipo de cliente entre contratos (`Conflicto_Tipo_Cliente = 1` en el **1.69%** de los casos).
* **Oportunidad Comercial Potencial:** La coexistencia de varios contratos de electricidad con diferentes potencias y discrepancias de registro en los sistemas abre la oportunidad de coordinar la gestión de sus cuentas, depurar la consistencia de sus datos internos y analizar sus datos técnicos contratados.
* **Líneas de Actuación Propuestas:**
  * *Gestión Coordinada de Cuentas:* Evaluar la asignación de un equipo o canal de atención coordinado para el seguimiento administrativo de las pólizas dispersas de la cuenta.
  * *Análisis y Depuración Contractual:* Revisar de forma sistemática los contratos para corregir las discrepancias detectadas por la variable `Conflicto_Tipo_Cliente` y analizar los datos de potencia contratada en relación con sus patrones de consumo.
* **Información Adicional Necesaria para la Validación:** Datos cualitativos de la actividad de los puntos de suministro, registros de consumo de potencia real (curva de demanda horaria), coste operativo de los equipos de soporte prioritario y análisis detallado de la consistencia regulatoria de los contratos.

### 8.3. Priorización Cualitativa y Prudente de las Estrategias

La determinación de prioridades para el despliegue de las líneas de actuación debe realizarse bajo criterios de prudencia metodológica, atendiendo al volumen físico de los clústeres y a la necesidad de información adicional, dado que la base de datos de partida carece de variables de rentabilidad, márgenes o comportamiento real de consumo:

1. **Prioridad Alta: Exploración sobre el Clúster 3 (C3 - Solo Luz Reciente)**
   * *Criterios:* Representa el **51.23% del portafolio** ($255,561$ clientes). Su condición de segmento de mayor escala hace que cualquier mejora en los canales de comunicación iniciales o en la viabilidad de la venta cruzada de gas tenga un impacto volumétrico directo sobre el desempeño de la compañía.
2. **Prioridad Media: Saneamiento y Gestión en el Clúster 5 (C5 - Alta Densidad de Contratos)**
   * *Criterios:* Integra al $5.47\%$ del total. Es prioritaria desde un enfoque de integridad operativa y de calidad de datos, dado que la depuración de la variable `Conflicto_Tipo_Cliente` (1.69% de inconsistencia interna en clasificación) permite corregir discrepancias en los sistemas administrativos y sentar las bases para una adecuada gestión coordinada.
3. **Prioridad Media-Baja: Venta Cruzada en el Clúster 2 (C2 - Solo Gas)**
   * *Criterios:* Representa el $4.02\%$ de la cartera. Aunque la oportunidad de venta cruzada de electricidad es clara, la viabilidad comercial y técnica está condicionada a la identificación preliminar de códigos CUPS eléctricos viables en las ubicaciones de gas.
4. **Prioridad Baja: Servicios Complementarios en el Clúster 0 (C0 - Solo Luz Antiguo) y Clúster 1 (C1 - Dual Intermedio)**
   * *Criterios:* Representan conjuntamente el $32.48\%$ del portafolio. Estas propuestas exigen verificar antes la receptividad real a servicios adicionales y comprobar los consumos reales para evitar costes comerciales de campaña que puedan no resultar eficaces.
5. **Prioridad Baja: Simplificación Administrativa en el Clúster 4 (C4 - Dual Multicontrato)**
   * *Criterios:* Afecta al $6.80\%$ del portafolio. Se posiciona en último lugar debido a la alta complejidad técnica y jurídica que conlleva modificar o integrar sistemas de facturación en el sector energético, requiriendo un análisis de viabilidad técnica muy exigente antes de tomar cualquier decisión de diseño.

### 8.4. Limitaciones y Cautelas de las Propuestas

Para preservar el rigor científico del estudio, se establecen las siguientes cautelas en la interpretación de los resultados:

* **Neutralidad del Registro Temporal:** Una antigüedad elevada (como en C0) no representa satisfacción, lealtad de marca o predisposición a la contratación de nuevos productos. Su permanencia en el registro puede responder a la inercia, desinterés por el mercado energético o barreras de salida no observadas en los datos.
* **Exclusión de Atribuciones Económicas:** Un mayor número de contratos por cliente (como en C4 y C5) o la clasificación dual del suministro no permiten deducir un mayor valor económico, márgenes comerciales superiores o mayor rentabilidad para la organización.
* **La Variable Conflicto no es una Incidencia Operativa:** Se subraya que la variable `Conflicto_Tipo_Cliente` identifica exclusivamente inconsistencias internas en la clasificación de la tipología del cliente entre distintos contratos en el sistema. No debe interpretarse en ningún caso como quejas, reclamaciones reales de facturación, problemas de servicio o descontento del cliente.
* **La Variabilidad de Potencia no representa un Fallo:** La presencia de diversos códigos de potencia en C5 responde a las necesidades de suministro físico de la cuenta. No implica un fallo técnico en las instalaciones ni presupone que los contratos puedan fusionarse técnica o legalmente.
* **Viabilidad Contractual y Regulatoria:** Cualquier propuesta que contemple la unificación de contratos de suministro de luz y gas o la simplificación administrativa está sujeta a rigurosas validaciones regulatorias, normativas del mercado y viabilidad de los sistemas de facturación internos.

### 8.5. Evaluación y Conclusión del Capítulo

Como fase futura de desarrollo externa al presente TFM, las estrategias planteadas podrían evaluarse tras un periodo de prueba piloto mediante la medición de indicadores de impacto específicos. Para las acciones de venta cruzada y captación (C2, C3), los indicadores idóneos abarcarían la **tasa de conversión comercial (%)** y el coste marginal de adquisición. En las acciones de servicios complementarios (C0, C1), la evaluación se mediría según la **tasa de contratación adicional (%)** e ingresos promedio generados. Finalmente, para las propuestas administrativas y operativas (C4, C5), los indicadores idóneos serían la **tasa de corrección de discrepancias en sistemas (%)** y los tiempos de tramitación interna.

En conclusión, la segmentación mediante el algoritmo $K$-Means original de seis clústeres ofrece una taxonomía de valor descriptivo para la compañía, permitiendo perfilar los suministros según su volumen de contratos, antigüedad registrada y tipo de servicio. No obstante, para transformar estas observaciones descriptivas en decisiones comerciales definitivas, la organización deberá integrar obligatoriamente bases de datos auxiliares de consumo, márgenes y elegibilidad técnica, preservando siempre la cautela analítica recomendada en este estudio.

## 8. Diseño de Estrategias Comerciales Personalizadas por Clúster

### 8.1. Introducción y Objetivo del Capítulo

El objetivo de este capítulo es traducir los resultados analíticos y taxonómicos obtenidos en la caracterización de la cartera de clientes (Capítulo 7) en un conjunto de **líneas de actuación comerciales aplicadas**. La segmentación avanzada mediante el algoritmo $K$-Means ($K=6$) no debe interpretarse únicamente como un ejercicio de clasificación matemática, sino como una herramienta de soporte a las decisiones estratégicas que permite superar las aproximaciones comerciales masivas (*one-size-fits-all*).

En este contexto, las propuestas presentadas a continuación no constituyen verdades empíricas absolutas ni asumen un éxito financiero garantizado. Se formulan como **líneas de actuación potenciales y exploratorias** cuya viabilidad, rentabilidad y conveniencia técnica final quedan condicionadas a la incorporación de variables comerciales, financieras y operativas adicionales de las que no se dispone en el dataset de origen. El enfoque adoptado es deliberadamente prudente y estructurado, buscando mitigar riesgos operativos antes del despliegue de cualquier campaña en el mercado real.

### 8.2. Estrategias Comerciales Personalizadas para los Seis Clústeres

El análisis de la segmentación realizada permite estructurar el portafolio de clientes en seis agrupaciones diferenciadas. A continuación, se detalla el análisis aplicado de cada clúster, estructurando la información entre las características empíricas observadas, las oportunidades de exploración comercial, las líneas de actuación propuestas y la información externa requerida para validar la viabilidad de cualquier medida.

---

#### Clúster 0 (C0): Suministro Eléctrico Exclusivo con Registro Temporal Alto
* **Características Observadas en los Datos:** Este segmento representa a $84,100$ clientes ($16.86\%$ del total de la cartera), caracterizados por disponer exclusivamente de suministro de electricidad (`Solo Luz` = $100.00\%$) asociado a un único contrato por cliente y con el registro de permanencia temporal más elevado de la base de datos (mediana de **92.98 meses**; media de **92.68 meses**). No registran ninguna discrepancia de clasificación en sus cuentas (`Conflicto_Tipo_Cliente = 0`).
* **Oportunidad Comercial Potencial:** La prolongada trayectoria de registro de estos clientes monoproducto de electricidad abre la posibilidad teórica de evaluar su receptividad hacia la incorporación de servicios complementarios o coberturas del hogar, así como explorar la viabilidad de una oferta de gas natural para el desarrollo de una relación dual.
* **Líneas de Actuación Propuestas:**
  * *Estudio de Receptividad de Servicios Complementarios:* Explorar el diseño de propuestas de asistencia o protección de pagos que complementen el suministro eléctrico base.
  * *Sondeo de Conversión Dual:* Planteamiento de comunicaciones informativas sobre las condiciones y ventajas de tarifas unificadas de luz y gas comercializadas por la misma entidad.
* **Información Adicional Necesaria para la Validación:** La implementación real de estas acciones requiere consultar previamente bases de datos de consumo de energía real (kWh), la elegibilidad técnica de gas en los códigos postales de la muestra, el margen bruto promedio actual de las cuentas y sus preferencias de canal de comunicación.

---

#### Clúster 1 (C1): Suministro Dual Monocontrato con Registro Temporal Corto-Medio
* **Características Observadas en los Datos:** Integra a $77,927$ clientes ($15.62\%$ de la cartera) que cuentan con suministro dual unificado bajo una sola póliza contractual (`Dual` = $100.00\%$). Muestran una antigüedad temporal intermedia (mediana de **20.01 meses**; media de **32.85 meses**) y una tasa mínima de inconsistencias en la clasificación de datos ($0.22\%$).
* **Oportunidad Comercial Potencial:** Al estar consolidada la dualidad del servicio desde el origen, la oportunidad comercial radica en consolidar la posición de la marca mediante la exploración de servicios de eficiencia energética o tecnología complementaria.
* **Líneas de Actuación Propuestas:**
  * *Evaluación de Servicios de Eficiencia:* Oferta hipotética de estudios básicos de ahorro energético adaptados al consumo dual combinado.
  * *Sondeo de Servicios de Transición:* Planteamiento de propuestas exploratorias sobre viabilidad técnica de autoconsumo fotovoltaico o compatibilidad con sistemas de recarga eléctrica de vehículos.
* **Información Adicional Necesaria para la Validación:** Para determinar la viabilidad de estas propuestas, es indispensable incorporar datos externos sobre el histórico de consumo estacional detallado, las características estructurales de la edificación (vivienda unifamiliar o colectiva), la ubicación geográfica y la rentabilidad financiera neta de la cuenta dualizada.

---

#### Clúster 2 (C2): Suministro de Gas Exclusivo con Contrato Único
* **Características Observadas en los Datos:** Representa el segmento más pequeño con $20,047$ clientes ($4.02\%$ de la cartera), definidos por consumir únicamente gas natural (`Solo Gas` = $100.00\%$), presentar un patrón de contrato único en el $99.10\%$ de los casos y registrar una antigüedad media de **38.69 meses** (mediana de **26.68 meses**).
* **Oportunidad Comercial Potencial:** Al estar la relación comercial limitada al suministro de gas, existe la oportunidad de estudiar la viabilidad de una venta cruzada para captar el suministro eléctrico asociado a estos hogares.
* **Líneas de Actuación Propuestas:**
  * *Propuesta de Unificación Energética:* Presentación de ofertas personalizadas para la contratación adicional del suministro eléctrico bajo condiciones de descuento cruzado en la tarifa global.
* **Información Adicional Necesaria para la Validación:** Es necesario verificar con carácter previo la cobertura de distribución eléctrica, la existencia de un código CUPS eléctrico activo, la vigencia técnica del Boletín de Instalación y los márgenes financieros asociados a la captación de luz en este segmento.

---

#### Clúster 3 (C3): Suministro Eléctrico Exclusivo con Registro Temporal Corto
* **Características Observadas en los Datos:** Es el segmento mayoritario de la base de datos, con $255,561$ clientes ($51.23\%$ del total), caracterizados por tener suministro eléctrico exclusivo (`Solo Luz` = $100.00\%$), un único contrato asignado y la menor antigüedad media registrada en el modelo (mediana de **17.08 meses**; media de **20.77 meses**), con $0$ conflictos de clasificación.
* **Oportunidad Comercial Potencial:** Dada la dimensión del grupo y su corta permanencia en el registro, la oportunidad se centra en el asentamiento de la relación contractual y el análisis de la viabilidad de incorporar un segundo suministro (gas).
* **Líneas de Actuación Propuestas:**
  * *Exploración de Venta Cruzada de Gas:* Envío selectivo de propuestas de suministro de gas natural.
  * *Seguimiento de Acogida:* Implementación de canales de comunicación de soporte para resolver dudas sobre las primeras facturas de electricidad y mejorar el posicionamiento de marca.
* **Información Adicional Necesaria para la Validación:** Requiere la incorporación de datos sobre cobertura real de la red de gas natural en los códigos postales afectados, el canal de captación de origen de cada cliente, su volumen de consumo eléctrico y el coste de adquisición comercial admisible.

---

#### Clúster 4 (C4): Suministro Dual Multicontrato con Registro Temporal Medio
* **Características Observadas en los Datos:** Reúne a $33,932$ clientes ($6.80\%$ del total) con suministro dual (`Dual` = $100.00\%$), una antigüedad media de **42.23 meses** (mediana de **33.54 meses**) y un patrón de contratación múltiple (media de **2.18 contratos** por cliente, donde el $100.00\%$ posee 2 o más contratos).
* **Oportunidad Comercial Potencial:** La presencia de varios contratos de suministro dual asociados a una misma cuenta abre el espacio para estudiar opciones de simplificación administrativa y de gestión de facturación.
* **Líneas de Actuación Propuestas:**
  * *Estudio de Unificación de Cuentas:* Ofrecer como línea de exploración la unificación administrativa de la facturación en una sola cuenta y una única fecha de cargo bancario.
  * *Canal de Gestión Coordinada:* Diseñar una vía de atención unificada para la resolución de gestiones relativas a sus múltiples pólizas.
* **Información Adicional Necesaria para la Validación:** Es imprescindible comprobar la viabilidad técnica y legal en los sistemas comerciales para agrupar diferentes puntos de suministro bajo un mismo titular, verificar la coincidencia de los datos de contrato y evaluar el impacto del coste de consolidación de sistemas frente a la rentabilidad agregada del cliente.

---

#### Clúster 5 (C5): Suministro Eléctrico con Registro Temporal Medio y Alta Densidad Contractual
* **Características Observadas en los Datos:** Constituido por $27,296$ clientes ($5.47\%$ del total), mayoritariamente eléctricos (`Solo Luz` = $99.99\%$) con la mayor densidad de contratación media del modelo (**2.41 contratos** por cliente, registrando un caso extremo de hasta 121 contratos). Presenta una media de **1.73 potencias contratadas** distintas y la tasa más elevada de discrepancias en la clasificación del tipo de cliente entre contratos (`Conflicto_Tipo_Cliente = 1` en el **1.69%** de los casos).
* **Oportunidad Comercial Potencial:** La coexistencia de múltiples contratos eléctricos y variadas potencias contratadas, unida a la existencia de discrepancias internas de clasificación, sugiere la necesidad de depurar la consistencia de los datos en sistemas y explorar la adecuación de la capacidad técnica contratada.
* **Líneas de Actuación Propuestas:**
  * *Campaña de Saneamiento y Depuración de Datos:* Corrección proactiva de las inconsistencias identificadas por la variable `Conflicto_Tipo_Cliente` para asegurar la coherencia de las fichas de datos contractuales.
  * *Línea de Análisis de Potencia Contratada:* Ofrecer análisis orientados a evaluar si los niveles de potencia registrados se corresponden de forma óptima con sus patrones reales de demanda.
* **Información Adicional Necesaria para la Validación:** Exige la integración de las curvas de demanda de potencia cuartohorarias, la verificación de los registros reales de quejas operativas y reclamaciones técnicas en atención al cliente, y la especificación de los costes del personal especializado necesario para gestionar estas interacciones.

### 8.2. Estrategias Comerciales Personalizadas para los Seis Clústeres

#### Clúster 0 (C0): Cartera Eléctrica Monocontrato con Registro Temporal Alto
* **Características Relevantes:** Agrupa a $84,100$ clientes ($16.86\%$ del total) con suministro exclusivo de electricidad (`Solo Luz` = $100.00\%$), un único contrato por cliente y el registro de permanencia temporal más prolongado de la cartera (mediana de **92.98 meses**; media de **92.68 meses**), sin registros de discrepancias en la parametrización de sus cuentas.
* **Oportunidad Comercial Potencial:** Al tratarse de clientes con una prolongada trayectoria de facturación exclusiva en electricidad, existe la oportunidad de explorar su receptividad hacia servicios de valor añadido o coberturas complementarias de mantenimiento eléctrico, así como su potencial conversión a un modelo dual si disponen de suministro de gas contratado con otra comercializadora.
* **Estrategia Propuesta:**
  * *Campaña de Servicios de Asistencia:* Presentación de ofertas de servicios de asistencia técnica o protección de pagos, orientadas a asegurar el correcto funcionamiento de su instalación eléctrica.
  * *Sondeo de Conversión Dual:* Lanzamiento de comunicaciones informativas sobre las ventajas de unificar los servicios de luz y gas bajo una misma marca.
* **Información Necesaria para su Aplicación:** Antes de ejecutar la estrategia, es imprescindible disponer de:
  1. Historial de consumo real de energía (kWh) para dimensionar si el coste del servicio adicional es proporcional a su facturación.
  2. Datos de cobertura de gas natural en su zona geográfica.
  3. Ratios de rentabilidad y margen bruto actual de la cuenta para estructurar descuentos de captación viables.

---

#### Clúster 1 (C1): Cartera Dual Monocontrato con Registro Temporal Corto-Medio
* **Características Relevantes:** Compuesto por $77,927$ clientes ($15.62\%$ del total) que disponen de suministro dual unificado bajo una única póliza contractual (`Dual` = $100.00\%$), con un historial temporal moderado (mediana de **20.01 meses**; media de **32.85 meses**) y un volumen muy bajo de inconsistencias de clasificación ($0.22\%$).
* **Oportunidad Comercial Potencial:** Estos clientes ya consumen ambos vectores energéticos (electricidad y gas) de manera unificada. La oportunidad se centra en consolidar esta relación comercial mediante programas que refuercen la comodidad de la factura única y exploren su interés en la transición energética (como soluciones de autoconsumo fotovoltaico o aerotermia).
* **Estrategia Propuesta:**
  * *Programa de Optimización de Consumo:* Envío de informes personalizados sobre eficiencia energética y recomendaciones de ahorro, posicionando a la comercializadora como un socio de confianza.
  * *Ofertas de Transición Ecológica:* Evaluación del perfil de vivienda para ofrecer pre-diagnósticos de instalación de placas solares o puntos de recarga de vehículos eléctricos.
* **Información Necesaria para su Aplicación:** Se requiere recopilar:
  1. Curvas de carga horaria de electricidad y consumos estacionales de gas.
  2. Tipo de vivienda (unifamiliar o comunitaria) para verificar la viabilidad técnica de instalaciones solares.
  3. Margen comercial integrado de la cuenta dual y tarifas de la competencia para asegurar ofertas competitivas.

---

#### Clúster 2 (C2): Suministro de Gas Exclusivo con Contrato Único
* **Características Relevantes:** El segmento más reducido de la cartera con $20,047$ clientes ($4.02\%$ del total), caracterizados por consumir exclusivamente gas natural (`Solo Gas` = $100.00\%$) bajo un patrón estricto de contrato único ($99.10\%$) y una antigüedad intermedia (mediana de **26.68 meses**; media de **38.69 meses**).
* **Oportunidad Comercial Potencial:** La exclusividad del producto gas representa una oportunidad de crecimiento orgánico interno mediante la venta cruzada (*cross-selling*) del suministro eléctrico, convirtiendo estas cuentas en clientes duales.
* **Estrategia Propuesta:**
  * *Campaña de Dualización Eléctrica:* Presentación de una oferta de suministro de electricidad garantizando condiciones preferenciales o descuentos cruzados si deciden unificar ambos suministros con la compañía.
* **Información Necesaria para su Aplicación:** Es indispensable auditar:
  1. La elegibilidad técnica del punto de suministro eléctrico (existencia de un CUPS de luz activo en la misma dirección y Boletín de Instalación Eléctrica vigente).
  2. El consumo medio de gas para priorizar la acción sobre los hogares con mayor volumen de facturación energética global.
  3. El coste de adquisición de cliente (CAC) proyectado frente al valor de tiempo de vida (LTV) esperado de una cuenta dualizada.

---

#### Clúster 3 (C3): Cartera Eléctrica Monocontrato con Registro Temporal Corto
* **Características Relevantes:** Es el clúster mayoritario de la organización, que integra a $255,561$ clientes ($51.23\%$ del total) con suministro exclusivo de electricidad (`Solo Luz` = $100.00\%$), un único contrato por cliente y la menor antigüedad registrada en la base de datos (mediana de **17.08 meses**; media de **20.77 meses**).
* **Oportunidad Comercial Potencial:** Al ser el grupo más grande y de más reciente incorporación, la prioridad inicial consiste en asentar la relación comercial. Existe una clara oportunidad para la venta cruzada del suministro de gas para convertirlos en duales antes de que se cumpla su ciclo de maduración temporal en la compañía.
* **Estrategia Propuesta:**
  * *Campaña Temprana de Venta Cruzada de Gas:* Ofrecer tarifas promocionales de gas natural asociadas a su contrato eléctrico vigente.
  * *Canal de Bienvenida y Adaptación:* Implementar comunicaciones digitales de seguimiento sobre la primera factura de luz para reducir la incertidumbre del cliente novel.
* **Información Necesaria para su Aplicación:** Se requiere confirmar:
  1. La disponibilidad física de canalización de gas natural en los códigos postales del segmento.
  2. El canal de captación de origen (digital, presencial, telefónico) para personalizar la vía de comunicación de la oferta.
  3. Los niveles de precios ofertados actualmente por la competencia directa en su zona de residencia.

---

#### Clúster 4 (C4): Cartera Dual Multicontrato con Registro Temporal Medio
* **Características Relevantes:** Cuenta con $33,932$ clientes ($6.80\%$ del total) clasificados como duales en su totalidad (`Dual` = $100.00\%$), con una trayectoria intermedia (mediana de **33.54 meses**; media de **42.23 meses**) y caracterizados por poseer múltiples contratos (media de **2.18 contratos** por cliente, donde el $100.00\%$ posee 2 o más contratos).
* **Oportunidad Comercial Potencial:** La posesión de múltiples contratos duales sugiere una estructura de suministro distribuida (por ejemplo, clientes residenciales con segundas residencias o pequeños negocios). La oportunidad reside en simplificar su carga administrativa mediante la unificación de cuentas bajo un único interlocutor y un marco de facturación consolidada.
* **Estrategia Propuesta:**
  * *Programa de Facturación Consolidada:* Ofrecer la unificación de todos sus contratos en un único documento de cobro y una única fecha de cargo bancario.
  * *Propuesta de Gestor de Cuenta Multicontrato:* Canalizar sus consultas a través de un servicio de atención preferente para agilizar sus gestiones.
* **Información Necesaria para su Aplicación:** Es obligatorio validar:
  1. La viabilidad técnica y contractual del sistema de facturación para agrupar diferentes CUPS (puntos de suministro) bajo una misma cuenta de cobro sin penalizaciones.
  2. La coincidencia exacta en la titularidad jurídica de los distintos contratos.
  3. El margen de contribución financiero agregado de todos los contratos asociados al cliente.

---

#### Clúster 5 (C5): Cartera Eléctrica Multicontrato con Registro Temporal Medio y Alta Complejidad
* **Características Relevantes:** Compuesto por $27,296$ clientes ($5.47\%$ del total) con suministro casi exclusivo de electricidad (`Solo Luz` = $99.99\%$) y la mayor densidad de contratación de la cartera (media de **2.41 contratos** y casos extremos de hasta **121 contratos** por cliente). Presenta una elevada variabilidad técnica (media de **1.73 potencias contratadas** distintas) y el índice más alto de discrepancias de clasificación en los sistemas (`Conflicto_Tipo_Cliente = 1` en el **1.69%** de los registros).
* **Oportunidad Comercial Potencial:** Este perfil es característico de clientes institucionales, corporativos, pymes con varios locales o comunidades de propietarios. Su alta complejidad técnica y la existencia de inconsistencias de datos en sistemas abren una clara oportunidad para el saneamiento operativo de sus cuentas y la oferta de asesoría técnica especializada en optimización de potencias.
* **Estrategia Propuesta:**
  * *Servicio de Auditoría de Potencia Eléctrica:* Ofrecer un estudio gratuito de optimización de los niveles de potencia contratada en todas sus ubicaciones para generar ahorros en sus costes fijos.
  * *Campaña de Saneamiento Operativo:* Contactar activamente con el cliente para depurar los datos de clasificación y corregir las inconsistencias detectadas en la variable `Conflicto_Tipo_Cliente`.
* **Información Necesaria para su Aplicación:** Se requiere disponer de:
  1. Curvas de demanda de potencia máxima (cuartohorarias) para realizar un cálculo preciso de optimización y evitar penalizaciones por excesos de potencia.
  2. Datos cualitativos de la actividad del cliente (CNAE, tipo de empresa) para adaptar la propuesta técnica.
  3. Registro histórico real de quejas y reclamaciones operativas para verificar si las discrepancias de clasificación de sistemas están afectando a la satisfacción del cliente.

### 8.3. Análisis de Diferenciación entre Clústeres con Perfiles Similares

Para maximizar el rigor metodológico del TFM, es fundamental explicitar las fronteras de decisión comercial entre aquellos segmentos que comparten atributos cuantitativos cercanos:

* **Diferenciación entre C0 y C3 (Suministros de Solo Luz Monocontrato):**
  Aunque ambos segmentos representan clientes con un único contrato eléctrico, **C3 es un segmento de captación reciente** (mediana de $17.08$ meses) y de gran volumen ($51.23\%$ de la cartera), por lo que las acciones deben orientarse a la venta cruzada ágil y a asegurar su permanencia inicial. En contraste, **C0 representa una cartera con registro temporal alto** (mediana de $92.98$ meses); aquí, las acciones agresivas de venta cruzada de gas deben plantearse con cautela comercial, priorizando en su lugar la oferta de servicios de valor añadido (asistencia) que no alteren el equilibrio o la inercia del cliente.

* **Diferenciación entre C1 y C4 (Suministros Duales):**
  A pesar de que ambos grupos consumen luz y gas, **C1 cuenta con un único contrato unificado**, representando un perfil de gestión sumamente sencillo y optimizado. La estrategia aquí es de fidelización ecológica y tecnológica (autoconsumo). Por el contrario, **C4 representa un perfil multicontrato** (media de $2.18$ contratos); en este caso, el esfuerzo comercial debe centrarse en la consolidación y simplificación de su estructura administrativa (factura unificada, consolidación de cobros) para reducir la fricción operativa.

* **Diferenciación entre C4 y C5 (Segmentos Multicontrato de Alta Densidad):**
  La distinción radica en el vector energético de consumo. **C4 es un segmento puramente dual**, lo que facilita propuestas integrales de energía para múltiples viviendas o locales de un mismo titular. En cambio, **C5 es un segmento monoproducto eléctrico** con una complejidad técnica muy superior (múltiples potencias contratadas y alta tasa de conflictos de datos). La prioridad en C5 no es la simplificación de facturas duales, sino el saneamiento operativo de la consistencia de sus cuentas en sistemas y la consultoría de optimización de capacidad eléctrica contratada.

### 8.4. Priorización Razonada de las Propuestas Comerciales

Al no disponer de datos de margen financiero neto real, consumos exactos o costes unitarios de campaña, se propone una **priorización estratégica cualitativa basada en el volumen de clientes afectados y la viabilidad operativa inmediata**:

1. **Prioridad 1: Venta Cruzada sobre el Clúster 3 (C3 - Solo Luz Reciente)**
   * *Justificación:* Afecta al **51.23% de la cartera de clientes** ($255,561$ registros). Al ser el segmento mayoritario, cualquier incremento marginal en su tasa de conversión a dual tendrá un impacto directo muy significativo en el volumen de negocio global de la organización. La venta cruzada de gas sobre contratos eléctricos ya existentes es una práctica estándar del sector con canales de comercialización maduros.
2. **Prioridad 2: Optimización de Potencia y Saneamiento en el Clúster 5 (C5 - Multicontrato Eléctrico Complejo)**
   * *Justificación:* Aunque representa el $5.47\%$ del portafolio, el volumen de contratos acumulados de este segmento es elevado y registra la tasa de conflictos en sistemas más alta ($1.69\%$). Sanear las fichas de datos y ofrecer una optimización de potencias actúa como una estrategia defensiva de retención de cuentas de alta contratación, reduciendo las discrepancias internas de administración.
3. **Prioridad 3: Venta Cruzada sobre el Clúster 2 (C2 - Solo Gas)**
   * *Justificación:* Constituye el $4.02\%$ del total. Es prioritaria por su baja complejidad y alta probabilidad de éxito: comercializar electricidad a un cliente de gas es técnicamente viable si el CUPS está activo, y permite recuperar cuota en el vector eléctrico de forma ágil.
4. **Prioridad 4: Servicios de Asistencia en el Clúster 0 (C0 - Solo Luz Antiguo)**
   * *Justificación:* Representa el $16.86\%$ del total. La facilidad de implementación de campañas de mantenimiento es alta, pero se prioriza por detrás de C3 y C5 para evitar saturar mediante impactos publicitarios a una cartera madura que opera con estabilidad en el registro.
5. **Prioridad 5: Facturación Consolidada en el Clúster 4 (C4 - Dual Multicontrato)**
   * *Justificación:* Representa el $6.80\%$ del total. Aunque el potencial comercial es interesante, la viabilidad está fuertemente condicionada a la capacidad técnica y de desarrollo de sistemas de facturación de la organización para consolidar pólizas distintas, lo que eleva el coste y la complejidad de implantación inicial.

### 8.5. Limitaciones Metodológicas y Cautelas Comerciales

De acuerdo con los principios de prudencia de esta investigación, la dirección de la compañía debe observar las siguientes cautelas antes de implementar las acciones descritas:

* **Neutralidad del Registro Temporal (Antigüedad):** Una antigüedad elevada en el registro (como en C0) no demuestra empíricamente fidelidad, satisfacción del cliente ni alta predisposición de compra. Lanzar campañas comerciales intrusivas sobre carteras estables pero inactivas comercialmente podría reactivar la búsqueda de ofertas en competidores, provocando un efecto rebote de baja de contratos (*churn inducido*).
* **La Variable Conflicto no es una Queja:** Se reitera de forma taxativa que la variable `Conflicto_Tipo_Cliente` identifica exclusivamente una discrepancia formal de codificación de la cuenta en bases de datos. No representa quejas, reclamaciones reales, incidencias de facturación ni disputas de clientes, por lo que no debe utilizarse para evaluar la calidad del servicio de atención al cliente.
* **La Complejidad Eléctrica no es un Fallo Técnico:** La diversidad de códigos de potencia en C5 responde a la multiplicidad de locales o puntos de suministro asociados a un único titular. No constituye un fallo de red o error de instalación, sino una consecuencia lógica de la naturaleza del cliente (pymes, corporativos).
* **No Presuponer Unificación Contractual:** La consolidación administrativa de pólizas (C4) o el empaquetado de tarifas requiere la verificación exhaustiva de los marcos regulatorios vigentes del sector y de la coincidencia legal de las titularidades para evitar contingencias jurídicas.
* **Exclusión de Métricas Financieras:** Al no disponer de datos de consumo de energía, márgenes unitarios de contribución, ni tasas de abandono (*churn*), ninguna de las estrategias comerciales propuestas puede considerarse como rentable *a priori* de forma matemática.

### 8.6. Protocolo de Evaluación de Impacto Futuro

Para asegurar la mejora continua y el rigor de la implantación estratégica en fases posteriores de la organización, se propone evaluar el impacto de cada propuesta mediante los siguientes indicadores de desempeño una vez ejecutadas:

* **En Campañas de Conversión (C2, C3):**
  * *Tasa de Conversión Dual (%):* Porcentaje de clientes impactados que contratan el suministro complementario en un plazo de 90 días.
  * *Coste de Adquisición Marginal (CAC €):* Coste total de la campaña dividido entre el número de nuevos productos dualizados.
* **En Campañas de Valor Añadido (C0):**
  * *Tasa de Contratación de Asistencia (%):* Volumen de pólizas de mantenimiento suscritas sobre el total de contactos realizados.
  * *Margen Incremental de la Cuenta (LTV €):* Incremento en el valor de tiempo de vida del cliente tras deducir el coste de prestación del servicio técnico.
* **En Campañas Operativas y de Saneamiento (C4, C5):**
  * *Tasa de Resolución de Conflictos (%):* Porcentaje de cuentas del clúster C5 que logran un estado libre de discrepancias en el sistema de información tras la intervención.
  * *Tasa de Adhesión a la Facturación Consolidada (%):* Porcentaje de aceptación del programa de unificación de cobros en C4.
  * *Índice de Retención del Cliente (Retención %):* Evolución de la tasa de baja voluntaria (churn) de las cuentas unificadas frente al comportamiento del grupo de control histórico.

Este protocolo metodológico garantiza que cualquier acción futura sea evaluada con rigor científico, permitiendo a la organización cuantificar con precisión el retorno de la inversión de su segmentación de clientes.